In [19]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

class EnhancedPolynomialModel:
    def __init__(self):
        # Lattice QCD parameters (from papers and PDG)
        self.alpha_s_mz = 0.11803

        # Quark reference masses (GeV)
        self.mc_mc = 1.2735
        self.mb_mb = 4.188
        self.mu_2gev = 0.00216
        self.md_2gev = 0.00467
        self.ms_2gev = 0.093
        self.mt_mt = 172.76

        # Reference scales (GeV)
        self.mu_ref_light = 2.0
        self.mu_c = self.mc_mc
        self.mu_b = self.mb_mb
        self.mu_t = self.mt_mt
        self.mz = 91.1876

        # Polynomial degree (cubic)
        self.poly_degree = 3

        # Initialize polynomial coefficients: 4 coefficients each
        self.c_coeffs_up = np.ones(self.poly_degree + 1) * 0.1
        self.c_coeffs_down = np.ones(self.poly_degree + 1) * 0.1

        # Top quark special exponential enhancement
        self.top_enhancement_factor = 10.0
        self.top_exponent = 1.5

        # Geodesic lengths initial guess
        self.L_u = 0.1
        self.L_d = 0.2
        self.L_s = 0.5
        self.L_c = 1.0
        self.L_b = 2.0
        self.L_t = 3.0

        # Generation scaling factors
        self.gen_scale = [1.0, 1.0, 0.05]

        # CKM angles and CP phase
        self.theta_12 = 0.2
        self.theta_13 = 0.01
        self.theta_23 = 0.04
        self.delta_cp = 1.2


    def calculate_mass_geodesic(self, quark, n):
        h_q = self.h_params[quark]
        phi_q = self.phi_params[quark]
        cos_v_q = np.clip(h_q ** 0.5 / (self.c + self.a * np.cos(self.v_guess)), -1, 1)
        v_q = np.arccos(cos_v_q)
        phase = self.omega * n + phi_q
        log_mass = self.lambda_base + self.epsilon * np.cos(phase)
        return 10 ** log_mass


        # Experimental CKM matrix magnitudes (PDG 2022)
        self.ckm_exp = np.array([
            [0.97435, 0.22500, 0.00369],
            [0.22486, 0.97349, 0.04182],
            [0.00857, 0.04110, 0.99915]
        ])

        # Beta function coefficients for different flavors
        self.beta0_nf3 = (11.0 - 2.0/3.0 * 3.0) / 4.0
        self.beta1_nf3 = (102.0 - 38.0/3.0 * 3.0) / 16.0
        self.beta0_nf4 = (11.0 - 2.0/3.0 * 4.0) / 4.0
        self.beta1_nf4 = (102.0 - 38.0/3.0 * 4.0) / 16.0
        self.beta0_nf5 = (11.0 - 2.0/3.0 * 5.0) / 4.0
        self.beta1_nf5 = (102.0 - 38.0/3.0 * 5.0) / 16.0
        self.beta0_nf6 = (11.0 - 2.0/3.0 * 6.0) / 4.0
        self.beta1_nf6 = (102.0 - 38.0/3.0 * 6.0) / 16.0

        # Anomalous dimension coefficients
        self.gamma0 = 1.0
        self.gamma1_nf3 = (202.0/3.0 - 20.0/9.0 * 3.0) / 16.0
        self.gamma1_nf4 = (202.0/3.0 - 20.0/9.0 * 4.0) / 16.0
        self.gamma1_nf5 = (202.0/3.0 - 20.0/9.0 * 5.0) / 16.0
        self.gamma1_nf6 = (202.0/3.0 - 20.0/9.0 * 6.0) / 16.0

        # Quark info
        self.quark_info = {
            'u': {'type': 'up', 'generation': 1, 'ref_mass': self.mu_2gev, 'ref_scale': self.mu_ref_light},
            'd': {'type': 'down', 'generation': 1, 'ref_mass': self.md_2gev, 'ref_scale': self.mu_ref_light},
            's': {'type': 'down', 'generation': 2, 'ref_mass': self.ms_2gev, 'ref_scale': self.mu_ref_light},
            'c': {'type': 'up', 'generation': 2, 'ref_mass': self.mc_mc, 'ref_scale': self.mu_c},
            'b': {'type': 'down', 'generation': 3, 'ref_mass': self.mb_mb, 'ref_scale': self.mu_b},
            't': {'type': 'up', 'generation': 3, 'ref_mass': self.mt_mt, 'ref_scale': self.mu_t}
        }


        # Geodesic torus parameters
        self.c = 3.0  # Major radius
        self.a = 1.0  # Minor radius
        self.v_guess = 0.0  # Placeholder
        self.lambda_base = 0.5
        self.epsilon = 0.3
        self.omega = 3.88

        self.h_params = {
            'u': 0.5,
            'd': 0.6,
            's': 0.7,
            'c': 0.8,
            'b': 0.9,
            't': 1.0
        }

        self.phi_params = {
            'u': 0.0,
            'd': 0.3,
            's': 0.6,
            'c': 0.9,
            'b': 1.2,
            't': 1.5
        }

        self.is_optimized = False
        self.results = {}

    def calculate_mass(self, quark, L=None):
        info = self.quark_info[quark]
        coeffs = self.c_coeffs_up if info['type'] == 'up' else self.c_coeffs_down
        L = L if L is not None else getattr(self, f"L_{quark}")
        val = sum(c * L ** i for i, c in enumerate(coeffs))
        val *= self.gen_scale[info['generation'] - 1]

        if quark == 't':
            val += self.top_enhancement_factor * np.exp(self.top_exponent * L)
        return val

    def calculate_ckm_matrix(self):
        s12, c12 = np.sin(self.theta_12), np.cos(self.theta_12)
        s13, c13 = np.sin(self.theta_13), np.cos(self.theta_13)
        s23, c23 = np.sin(self.theta_23), np.cos(self.theta_23)
        delta = self.delta_cp
        ckm = np.array([
            [c12 * c13, s12 * c13, s13 * np.exp(-1j * delta)],
            [-s12 * c23 - c12 * s23 * s13 * np.exp(1j * delta),
             c12 * c23 - s12 * s23 * s13 * np.exp(1j * delta),
             s23 * c13],
            [s12 * s23 - c12 * c23 * s13 * np.exp(1j * delta),
             -c12 * s23 - s12 * c23 * s13 * np.exp(1j * delta),
             c23 * c13]
        ])
        return ckm

    def optimize_parameters(self):
        n = self.poly_degree + 1

        def objective(params):
            self.c_coeffs_up = params[0:n]
            self.c_coeffs_down = params[n:2 * n]
            self.top_enhancement_factor = params[2 * n]
            self.top_exponent = params[2 * n + 1]
            idx = 2 * n + 2
            self.L_u = params[idx]
            self.L_d = params[idx + 1]
            self.L_s = params[idx + 2]
            self.L_c = params[idx + 3]
            self.L_b = params[idx + 4]
            self.L_t = params[idx + 5]
            idx += 6
            self.gen_scale = params[idx:idx + 3]
            idx += 3
            self.theta_12 = params[idx]
            self.theta_13 = params[idx + 1]
            self.theta_23 = params[idx + 2]
            self.delta_cp = params[idx + 3]

            errors = {}
            for q in self.quark_info:
                ref = self.quark_info[q]['ref_mass']
                pred = self.calculate_mass(q)
                errors[q] = ((pred - ref) / ref) ** 2

            ckm = self.calculate_ckm_matrix()
            ckm_mag = np.abs(ckm)
            sq_err = (ckm_mag - self.ckm_exp) ** 2

            small_indices = [(0, 2), (1, 2), (2, 0), (2, 1)]
            ckm_weighted_err = 0.
            for i_ in range(3):
                for j_ in range(3):
                    w = 50.0 if (i_, j_) in small_indices else 1.0
                    ckm_weighted_err += w * sq_err[i_, j_]

            total_error = (errors['u'] + errors['d'] + errors['s'] +
                           5 * errors['c'] + 5 * errors['b'] + 20 * errors['t'] +
                           15 * ckm_weighted_err)

            reg = 0.01 * (
                np.sum(self.c_coeffs_up ** 2) + np.sum(self.c_coeffs_down ** 2) +
                self.top_enhancement_factor ** 2 + self.top_exponent ** 2 +
                np.sum((np.array([self.L_u, self.L_d, self.L_s, self.L_c, self.L_b, self.L_t]) -
                        np.array([0.1, 0.2, 0.5, 1.0, 2.0, 3.0])) ** 2)
            )
            return total_error + reg

        initial_guess = np.concatenate([
            self.c_coeffs_up,
            self.c_coeffs_down,
            [self.top_enhancement_factor, self.top_exponent],
            [self.L_u, self.L_d, self.L_s, self.L_c, self.L_b, self.L_t],
            self.gen_scale,
            [self.theta_12, self.theta_13, self.theta_23, self.delta_cp]
        ])

        bounds = []
        bounds.extend([(0.001, 10.0) for _ in range(n)])  # up coeffs
        bounds.extend([(0.001, 10.0) for _ in range(n)])  # down coeffs
        bounds.append((0.1, 1000.0))  # top_enh_factor
        bounds.append((0.1, 5.0))  # top_exp
        bounds.append((0.005, 1.0))  # L_u
        bounds.append((0.005, 1.0))  # L_d
        bounds.append((0.05, 1.5))  # L_s
        bounds.append((0.3, 3.0))  # L_c
        bounds.append((1.0, 5.0))  # L_b
        bounds.append((2.0, 6.0))  # L_t
        bounds.append((0.001, 10.0))  # gen_scale 1st
        bounds.append((0.001, 3.0))  # gen_scale 2nd
        bounds.append((0.001, 0.5))  # gen_scale 3rd
        bounds.append((0.1, 0.3))  # th_12
        bounds.append((0.001, 0.05))  # th_13
        bounds.append((0.01, 0.1))  # th_23
        bounds.append((0.0, 2 * np.pi))  # delta_cp

        # Increased maxfun and maxiter options for better convergence
        result = minimize(objective, initial_guess, bounds=bounds, method="L-BFGS-B",
                          options={"maxfun": 50000, "maxiter": 10000})

        # Unpack optimized results
        self.c_coeffs_up = result.x[0:n]
        self.c_coeffs_down = result.x[n:2 * n]
        self.top_enhancement_factor = result.x[2 * n]
        self.top_exponent = result.x[2 * n + 1]
        idx = 2 * n + 2
        self.L_u = result.x[idx]
        self.L_d = result.x[idx + 1]
        self.L_s = result.x[idx + 2]
        self.L_c = result.x[idx + 3]
        self.L_b = result.x[idx + 4]
        self.L_t = result.x[idx + 5]
        idx += 6
        self.gen_scale = result.x[idx:idx + 3]
        idx += 3
        self.theta_12 = result.x[idx]
        self.theta_13 = result.x[idx + 1]
        self.theta_23 = result.x[idx + 2]
        self.delta_cp = result.x[idx + 3]

        masses = {}
        errors = {}
        for q in self.quark_info:
            ref = self.quark_info[q]['ref_mass']
            pred = self.calculate_mass(q)
            masses[q] = pred
            errors[q] = abs((pred - ref) / ref) * 100

        ckm = self.calculate_ckm_matrix()
        ckm_mag = np.abs(ckm)
        ckm_err_pct = np.abs((ckm_mag - self.ckm_exp) / self.ckm_exp) * 100

        self.results = {
            "c_coeffs_up": self.c_coeffs_up,
            "c_coeffs_down": self.c_coeffs_down,
            "top_enhancement_factor": self.top_enhancement_factor,
            "top_exponent": self.top_exponent,
            "L_u": self.L_u,
            "L_d": self.L_d,
            "L_s": self.L_s,
            "L_c": self.L_c,
            "L_b": self.L_b,
            "L_t": self.L_t,
            "gen_scale": self.gen_scale,
            "theta_12": self.theta_12,
            "theta_13": self.theta_13,
            "theta_23": self.theta_23,
            "delta_cp": self.delta_cp,
            "masses": masses,
            "errors": errors,
            "ckm": ckm_mag,
            "ckm_exp": self.ckm_exp,
            "ckm_errors": ckm_err_pct,
            "success": result.success,
            "message": result.message,
        }
        self.is_optimized = True
        return self.results


if __name__ == "__main__":
    model = EnhancedPolynomialModel()
    results = model.optimize_parameters()

    print("Optimization success:", results["success"])
    print("Message:", results["message"])
    print("Top enhancement factor:", results["top_enhancement_factor"])
    print("Top enhancement exponent:", results["top_exponent"])
    print(
        "CKM angles (deg):",
        np.degrees(results["theta_12"]),
        np.degrees(results["theta_13"]),
        np.degrees(results["theta_23"]),
    )
    print("Masses at reference scales (GeV):")
    for q in model.quark_info:
        print(f" {q}: {results['masses'][q]:.6f}, error {results['errors'][q]:.3f} %")

    print("CKM matrix magnitudes:")
    print(results["ckm"])

AttributeError: 'EnhancedPolynomialModel' object has no attribute 'quark_info'